# Rebuilding a Face Using Fourier Transform

**Author:** Muhammad Bilal Azam  
**Date:** April 2026  

---

This notebook is a step-by-step tutorial showing how a grayscale image can be decomposed into frequency components and reconstructed using the **2D Discrete Fourier Transform (DFT)**.

The goal is not just to run the code, but to understand what each step is doing.

By the end, we will see that an image is not only a grid of pixels — it can also be viewed as a weighted sum of wave patterns with different frequencies, strengths, and spatial arrangements.

## What you need

Install the required packages if needed:

```bash
pip install numpy matplotlib pillow
```

You also need one image file. Put it in the same folder as this notebook and name it:

```text
face.jpg
```

You can use any image, but a face image makes the reconstruction visually interesting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.animation import FuncAnimation
from matplotlib.animation import PillowWriter # used to save GIF files
from matplotlib.animation import FFMpegWriter # used to save MP4 files
from IPython.display import HTML # used to display animations in Jupyter

## Step 1 — Load the image

We first load the image and convert it to grayscale.

A grayscale image is simpler than a color image because each pixel has only one number: its brightness. The value is usually between 0 and 255:

- 0 means black
- 255 means white
- values in between represent different shades of gray

Mathematically, we can think of the image as a 2D function:

$$
f(x,y)
$$

where $x$ and $y$ are pixel coordinates, and $f(x,y)$ is the brightness value at that pixel.

This is the object we will transform using the Fourier Transform.

In [ ]:
# Path to the image file.
# The image should be in the same folder as this notebook.
image_path = "face.jpg"

# Open the image and convert it to grayscale.
# "L" means luminance mode in PIL: one brightness value per pixel.
img = Image.open(image_path).convert("L")

# Convert the PIL image into a NumPy array.
# This gives us a 2D array of pixel values.
img = np.array(img)

# Print basic information about the image.
# Shape gives the number of rows and columns: (height, width).
print("Image shape:", img.shape)

# Pixel range tells us the minimum and maximum brightness values.
print("Pixel value range:", img.min(), "to", img.max())

# Display the image.
plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="gray")
plt.title("Original Image")
plt.axis("off")
plt.show()

### Checkpoint

At this point, the image is just a NumPy array.

Each entry in this array represents the brightness of a pixel.

If the image shape is $(N, M)$, then we have $N \times M$ brightness values.

So far, we are working in **pixel space**:
- each value corresponds to a location in the image

Next, we will switch to a completely different representation: **frequency space**.

## Step 2 — Compute the 2D Fourier Transform

Now we move from **pixel space** to **frequency space**.

Instead of describing the image using pixel values, we describe it as a sum of wave-like patterns with different frequencies.

The 2D Discrete Fourier Transform (DFT) is defined as:

$$
F(u,v)=\sum_{x=0}^{N-1}\sum_{y=0}^{M-1} f(x,y)\,
e^{-2\pi i\left(\frac{ux}{N}+\frac{vy}{M}\right)}
$$

Here:

- $f(x,y)$ is the original image  
- $F(u,v)$ is the Fourier coefficient at frequency $(u,v)$  
- each coefficient represents a sinusoidal pattern with:
  - a frequency  
  - an amplitude  
  - a phase  

So instead of storing pixel intensities, we now store how strongly each wave pattern contributes to the image.

---

### What does this mean intuitively?

You can think of the image as being built from many overlapping waves:

- low-frequency waves → smooth variations (large-scale structure)  
- high-frequency waves → rapid changes (edges and fine details)  

The Fourier Transform extracts all of these components.

---

### Computing it in Python

NumPy provides an efficient implementation of the Fast Fourier Transform (FFT).

After computing the transform, each entry in `F_shifted` is a **complex number**.  
This complex value encodes:

- the **magnitude** → how strong that frequency component is  
- the **phase** → how that pattern is positioned in the image  

In [ ]:
# Compute the 2D Fourier Transform of the image.
# This converts the image from pixel space to frequency space.
F = np.fft.fft2(img)

# Shift the zero-frequency component to the center of the spectrum.
# By default, the zero frequency is at the top-left corner.
# fftshift moves it to the center, which makes visualization and masking easier.
F_shifted = np.fft.fftshift(F)

# Print basic information about the Fourier array.
# Shape should be same as the image.
print("Fourier array shape:", F_shifted.shape)

# The data type is complex because each coefficient has real + imaginary parts.
print("Fourier array dtype:", F_shifted.dtype)

## Step 3 — Visualize the Fourier spectrum

The Fourier coefficients are complex numbers. To visualize them, we look at their magnitude:

$$
|F(u,v)|
$$

This tells us how strong each frequency component is.

However, the magnitude values can vary over several orders of magnitude.  
A few frequencies (especially low frequencies) are very strong, while most others are much weaker.

To make both strong and weak components visible, we plot:

$$
\log(1 + |F(u,v)|)
$$

The logarithm compresses the dynamic range, so smaller values are not completely hidden. We add 1 inside the logarithm to avoid issues when $|F(u,v)| = 0$, since $\log(0)$ is undefined.

---

### What are we looking at?

This image is called the **Fourier spectrum**.

Each point corresponds to a frequency component:

- the **center** represents low frequencies (slow variations in the image)  
- points farther from the center represent higher frequencies (finer details)  

---

### How to interpret this plot

- The **bright spot at the center**  
  → strong low-frequency components  
  → corresponds to overall brightness and smooth structure  

- The **fading intensity outward**  
  → high frequencies have smaller magnitude  

- The **cross-like pattern** (if visible)  
  → indicates dominant horizontal and vertical structures in the image  

---

### Why this matters

This visualization shows where most of the “energy” of the image lies in frequency space.

In the next step, we will use this idea to selectively keep only certain regions of the spectrum.

In [ ]:
# Compute the magnitude of the Fourier coefficients.
# np.abs() gives the magnitude of complex numbers.
magnitude = np.abs(F_shifted)

# Apply log scaling to compress dynamic range.
# Adding 1 avoids log(0).
spectrum = np.log(1 + magnitude)

# Display the spectrum.
plt.figure(figsize=(6, 5))
plt.imshow(spectrum, cmap="gray")
plt.title("Fourier Spectrum: log(1 + |F|)")
plt.axis("off")
plt.show()

### What to notice

Looking at this spectrum:

- There is a strong bright spot at the center  
- The intensity gradually decreases as we move outward  
- Some faint directional patterns may also be visible  

The bright center indicates that **low-frequency components dominate this image**.

This means:
- most of the image’s energy is concentrated in smooth, slowly varying structures  
- higher-frequency components (edges and textures) are present, but weaker  

---

### Why this is important for the experiment

This suggests that we might be able to keep only a small region around the center and still retain the main structure of the image.

In the next step, we will test exactly that by selectively removing high-frequency components.

## Step 4 — Sanity check: reconstruct without removing anything

Before we filter the spectrum, we should verify that the inverse transform gives back the original image.

The inverse DFT is:

$$
f(x,y)=\frac{1}{NM}\sum_{u=0}^{N-1}\sum_{v=0}^{M-1} F(u,v)
e^{2\pi i\left(\frac{ux}{N}+\frac{vy}{M}\right)}
$$

This equation says that if we keep all Fourier coefficients and add all the wave patterns back together, we should recover the original image.

NumPy handles the normalization internally.

This step is a useful sanity check: before we start removing frequencies, we confirm that the Fourier transform and inverse Fourier transform are working correctly.

In [ ]:
# Move the zero-frequency component back to its original position.
# We shifted it to the center earlier using fftshift, so now we undo that.
F_unshifted = np.fft.ifftshift(F_shifted)

# Apply the inverse 2D Fourier Transform.
# This reconstructs the image from all Fourier coefficients.
img_back_complex = np.fft.ifft2(F_unshifted)

# The result should be real-valued because the original image was real.
# Tiny imaginary parts can appear due to numerical roundoff, so we take the real part.
img_back = np.real(img_back_complex)

# Display the reconstructed image.
plt.figure(figsize=(5, 5))
plt.imshow(img_back, cmap="gray", vmin=0, vmax=255)
plt.title("Reconstructed Image: No Frequencies Removed")
plt.axis("off")
plt.show()

# Compare the reconstruction to the original image.
# The error should be extremely small, around machine precision.
max_error = np.max(np.abs(img.astype(float) - img_back))
print("Maximum reconstruction error:", max_error)

### Checkpoint

The reconstructed image should look identical to the original.

The numerical error should be extremely small (on the order of $10^{-12}$ or smaller).  
This tiny difference comes from floating-point precision and is expected.

This confirms an important point:

> The Fourier transform itself does **not** lose information.  
> It is simply a different representation of the same image.

Any loss of information in later steps will come from **deliberately removing frequency components**, not from the transform itself.

## Step 5 — Main experiment: keeping only a small block of low frequencies

Now we are ready for the main experiment.

So far, we have taken an image and converted it from **pixel space** into **frequency space** using the 2D Fourier transform. In pixel space, the image is described by brightness values at different positions. In frequency space, the same image is described as a combination of many wave-like patterns.

The purpose of this step is to ask a simple question:

> What happens if we reconstruct the image using only a small part of its Fourier spectrum?

More specifically, we will keep only the frequencies near the center of the Fourier spectrum and remove the rest.

This operation is called a **low-pass filter**, because it allows **low frequencies** to pass through while removing **high frequencies**.

Low frequencies correspond to smooth, slowly changing features of the image, such as:

- broad shapes
- overall lighting
- smooth regions
- large-scale structure

High frequencies correspond to rapidly changing features, such as:

- sharp edges
- fine details
- texture
- small variations from pixel to pixel

So by keeping only low frequencies, we expect the reconstructed image to look smoother and blurrier than the original.

---

### The key idea

The 2D discrete Fourier transform of an image can be written as:

$$
F(u,v)
=
\sum_{x=0}^{N-1}
\sum_{y=0}^{M-1}
f(x,y)\,
e^{-2\pi i\left(\frac{ux}{N}+\frac{vy}{M}\right)}
$$

Here:

- $f(x,y)$ is the original image in pixel space
- $F(u,v)$ is the Fourier coefficient at frequency index $(u,v)$
- $N$ and $M$ are the image dimensions
- each pair $(u,v)$ corresponds to a 2D wave pattern

The inverse transform reconstructs the image by adding all these frequency components back together:

$$
f(x,y)
=
\sum_{u=0}^{N-1}
\sum_{v=0}^{M-1}
F(u,v)\,
e^{2\pi i\left(\frac{ux}{N}+\frac{vy}{M}\right)}
$$

If we keep all Fourier coefficients, we recover the original image exactly, apart from tiny numerical precision errors.

But in this experiment, we will not keep all coefficients. We will keep only a selected block near the center of the spectrum.

That means the reconstruction is no longer exact. It becomes a **truncated Fourier expansion**, where only a subset of frequency components contribute to the image.

$$
f_{\text{rec}}(x,y)
\approx
\sum_{(u,v)\in \mathcal{S}}
F(u,v)\,
e^{2\pi i\left(\frac{ux}{N}+\frac{vy}{M}\right)}
$$

where $\mathcal{S}$ is the set of Fourier modes that we decide to keep.

---

### Why the center of the spectrum matters

After computing the Fourier transform, we use `fftshift`:

```python
F_shifted = np.fft.fftshift(F)
```

This moves the zero-frequency component to the center of the array.

The zero-frequency component is also called the **DC component**. It represents the average brightness of the image.

After `fftshift`, the spectrum is arranged like this:

- center of the spectrum → lowest frequencies
- farther from the center → higher frequencies
- outer regions → highest frequencies

So when we keep a square block around the center, we are keeping the lowest-frequency Fourier modes.

This is why the operation behaves like a low-pass filter.

---

### The mask

To select which Fourier coefficients to keep, we create a mask.

A mask is just an array of zeros and ones with the same shape as the Fourier spectrum:

- value 1 means: keep this Fourier coefficient
- value 0 means: remove this Fourier coefficient

The important line is:

```python
mask[crow-keep:crow+keep, ccol-keep:ccol+keep] = 1
```

Here:

- `crow` is the row index of the center of the spectrum
- `ccol` is the column index of the center of the spectrum
- `keep` controls how far we extend away from the center

This line sets a square block around the center of the mask equal to 1. Everything outside that square remains 0.

Then we multiply the Fourier spectrum by this mask:

```python
F_filtered = F_shifted * mask
```

Mathematically, this means:

$$
F_{\text{filtered}}(u,v)
=
F(u,v)\,M(u,v)
$$

where $M(u,v)$ is the mask:

$$
M(u,v)=
\begin{cases}
1, & (u,v) \text{ is inside the selected square} \\
0, & (u,v) \text{ is outside the selected square}
\end{cases}
$$

The mask does not change the values of the selected coefficients. It simply keeps some and sets the rest to zero.

This is equivalent to multiplying the Fourier spectrum by a window function in frequency space. In signal processing terms, we are explicitly limiting the **bandwidth** of the image.

---

### What does `keep` mean?

The parameter `keep` controls the size of the square region we keep around the center of the spectrum.

The selected region extends:

- `keep` pixels upward from the center
- `keep` pixels downward from the center
- `keep` pixels left from the center
- `keep` pixels right from the center

So the retained square has approximate size:

$$
(2\cdot \text{keep}) \times (2\cdot \text{keep})
$$

For example:

- `keep = 5` keeps a square of about $10 \times 10$ Fourier coefficients
- `keep = 20` keeps a square of about $40 \times 40$ Fourier coefficients
- `keep = 80` keeps a square of about $160 \times 160$ Fourier coefficients

As `keep` increases, the square grows outward from the center.

This means that larger values of `keep` include more frequency components.

---

### The selected frequency set

We can describe the selected square mathematically.

Let $(u_0,v_0)$ be the center of the shifted Fourier spectrum. Then the retained frequency set is approximately:

$$
\mathcal{S}_{\text{keep}}
=
\left\{
(u,v)
\;\middle|\;
|u-u_0| \le \text{keep},
\quad
|v-v_0| \le \text{keep}
\right\}
$$

This means we keep only those frequency indices whose row and column positions are within `keep` pixels of the center.

The filtered reconstruction is therefore:

$$
f_{\text{rec}}(x,y)
=
\sum_{(u,v)\in \mathcal{S}_{\text{keep}}}
F(u,v)\,
e^{2\pi i\left(\frac{ux}{N}+\frac{vy}{M}\right)}
$$

So we are not reconstructing the image from all Fourier modes. We are reconstructing it from a truncated set of low-frequency modes.

That is the mathematical meaning of this experiment.

---

### How `keep` changes the reconstruction

Changing `keep` changes the bandwidth of the reconstruction.

A small `keep` means we use only a very narrow band of low frequencies. The result is very smooth because sharp changes require high-frequency components.

A larger `keep` means we allow more frequencies into the reconstruction. The image becomes sharper because we are adding back modes that describe edges and finer structure.

In practice, the progression looks like this:

- very small `keep`  
  The reconstruction contains only the coarsest structure. The image may look like a blurry blob.

- moderate `keep`  
  Large shapes become visible. For a face image, the head shape and major facial regions may start to appear.

- large `keep`  
  Edges, contours, and fine details return. The reconstruction becomes closer to the original image.

So `keep` acts like a detail knob.

Turning it up allows more detail to enter the reconstruction.

---

### Why this works

Natural images usually contain a lot of energy in low frequencies.

That means the broad structure of an image is often captured by relatively few low-frequency components.

This is why a blurred version of the image can still be recognizable.

High frequencies are still important, but they mostly refine the image by adding:

- sharp boundaries
- small texture
- fine edges
- local contrast

This is also why image compression methods can often discard or reduce some high-frequency information while preserving the main content of the image.

---

### Square mask versus circular mask

In this tutorial, we use a square mask because it is simple to implement with array slicing:

```python
mask[crow-keep:crow+keep, ccol-keep:ccol+keep] = 1
```

A square mask keeps frequencies satisfying separate limits in the horizontal and vertical directions:

$$
|u-u_0| \le \text{keep},
\qquad
|v-v_0| \le \text{keep}
$$

A circular mask would instead keep frequencies based on distance from the center:

$$
\sqrt{(u-u_0)^2 + (v-v_0)^2} \le R
$$

A circular mask is often more natural if we want a radially symmetric frequency cutoff.

However, for the purpose of this experiment, the square mask is perfectly fine. It clearly shows how adding more low-frequency components gradually improves the reconstruction.

---

### What to expect before running the code

Before running the next cell, we should expect the following:

1. The original image will be transformed into frequency space.
2. A square block around the center of the spectrum will be kept.
3. Everything outside that block will be removed.
4. The inverse Fourier transform will reconstruct an image from only the retained frequencies.
5. Small `keep` values will produce blurry images.
6. Larger `keep` values will produce sharper images.

This is the central experiment of the tutorial.

We are not changing the image directly in pixel space. Instead, we are changing which Fourier modes are allowed to contribute to the reconstruction.

That is why this experiment is a direct visual demonstration of how an image can be built from frequency components.

### Visualizing the mask

Before applying the mask, it is helpful to visualize which part of the frequency spectrum we are keeping.

In the plot below:
- white region → frequencies we keep  
- black region → frequencies we remove  

In [ ]:
# Get image dimensions
rows, cols = img.shape

# Compute the center of the Fourier spectrum
# (this is where the zero-frequency component is located after fftshift)
crow, ccol = rows // 2, cols // 2

# Parameter controlling how many frequencies we keep
keep = 20

# Create a mask initialized with zeros (remove everything by default)
mask = np.zeros((rows, cols))

# Set a square region around the center to 1 (keep these frequencies)
mask[crow-keep:crow+keep, ccol-keep:ccol+keep] = 1

# Visualize the mask
plt.figure(figsize=(5, 5))
plt.imshow(mask, cmap="gray")

plt.title(f"Frequency Mask (keep = {keep})")
plt.axis("off")
plt.show()

The white square shows the region of frequency space that will be retained during reconstruction. Everything outside this region will be removed.

### Apply the mask and reconstruct

Now we multiply the Fourier spectrum by the mask.

This keeps the coefficients inside the white square and sets everything outside the square to zero.

After that, we apply the inverse Fourier transform. The result is an image reconstructed using only the selected low-frequency components.

In [ ]:
# Apply the mask in frequency space.
# Frequencies inside the white square are kept.
# Frequencies outside the square are set to zero.
F_filtered = F_shifted * mask

# Shift the zero-frequency component back to its original FFT position.
F_filtered_unshifted = np.fft.ifftshift(F_filtered)

# Reconstruct the image using the inverse 2D Fourier Transform.
img_rec_complex = np.fft.ifft2(F_filtered_unshifted)

# The reconstructed image should be real-valued.
# Tiny imaginary parts may appear due to numerical precision.
img_rec = np.real(img_rec_complex)

# Display the low-pass reconstructed image.
plt.figure(figsize=(5, 5))
plt.imshow(img_rec, cmap="gray", vmin=0, vmax=255)
plt.title(f"Reconstructed Image with keep = {keep}")
plt.axis("off")
plt.show()

### What happened?

With `keep = 20`, the image should look blurred. This is expected because we removed high-frequency components, which are responsible for edges, texture, and fine detail.

## Step 6 — Reconstruct using different numbers of frequencies

Now we repeat the same reconstruction for several values of `keep`.

This lets us see how the image changes as we gradually allow more frequency components into the reconstruction.

- Small `keep` → small square around the spectrum center → very few low-frequency components  
- Large `keep` → larger square → more frequencies included → more detail restored  

This is the main visual result of the experiment.

In [ ]:
# Different values of keep to test.
# Each value controls the half-width of the square frequency window.
keep_values = [5, 20, 40, 80, 150, 220, 300]

# Number of reconstructions we will generate.
n = len(keep_values)

# Arrange plots in a grid.
# We add 2 extra panels: one for the original image and one for the spectrum.
cols_plot = 4
rows_plot = int(np.ceil((n + 2) / cols_plot))

plt.figure(figsize=(4 * cols_plot, 4 * rows_plot))

index = 1

# Panel 1: original image
plt.subplot(rows_plot, cols_plot, index)
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.title("Original")
plt.axis("off")

# Panel 2: Fourier spectrum
index += 1
spectrum = np.log(1 + np.abs(F_shifted))

plt.subplot(rows_plot, cols_plot, index)
plt.imshow(spectrum, cmap="gray")
plt.title("Spectrum")
plt.axis("off")

# Panels 3 onward: reconstructions for different keep values
for keep in keep_values:

    # Create a new mask for this keep value.
    mask = np.zeros((rows, cols))

    # Keep a square block centered on the low-frequency region.
    mask[crow-keep:crow+keep, ccol-keep:ccol+keep] = 1

    # Apply the mask in frequency space.
    F_filtered = F_shifted * mask

    # Reconstruct image from the filtered spectrum.
    img_rec_complex = np.fft.ifft2(np.fft.ifftshift(F_filtered))

    # Take the real part; small imaginary values come from numerical roundoff.
    img_rec = np.real(img_rec_complex)

    # Display this reconstruction.
    index += 1
    plt.subplot(rows_plot, cols_plot, index)
    plt.imshow(img_rec, cmap="gray", vmin=0, vmax=255)
    plt.title(f"keep = {keep}")
    plt.axis("off")

plt.tight_layout()
plt.show()

### What to notice

As `keep` increases, the reconstruction becomes sharper.

For very small values such as `keep = 5`, only the broadest intensity patterns survive, so the result looks like a blurry version of the image.

By `keep = 20` or `keep = 40`, the face is already recognizable, even though many details are missing.

For larger values such as `keep = 150` or above, edges, texture, and fine details return, and the reconstruction becomes very close to the original image.

This shows that low frequencies carry the global structure of the image, while higher frequencies mainly restore sharpness and fine detail.

## Step 7 — Try random frequency sampling

Now let us try something different.

Instead of keeping a structured region (the center), we randomly keep a fraction of all frequencies.

This means:
- each frequency is kept with some probability
- the selection is independent of its position in the spectrum

This is still a valid Fourier reconstruction, but now the selection does **not respect the structure of the image**.

---

### What are we testing here?

In the previous step, we kept a **low-frequency block**, which worked well.

Now we test a different idea:

> What if we keep the same number of frequencies, but choose them randomly?

If all frequencies were equally important, this should work just as well.

But natural images are not like that.

In [ ]:
# Fractions of frequencies to keep.
# For example, 0.01 means we keep ~1% of all frequency components.
fractions = [0.001, 0.005, 0.01, 0.03, 0.07, 0.10, 0.20, 0.30,
             0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]

# Number of different reconstructions we will create
n = len(fractions)

# Layout for plotting
# +1 because we include the original image
cols_plot = 4
rows_plot = int(np.ceil((n + 1) / cols_plot))

plt.figure(figsize=(4 * cols_plot, 4 * rows_plot))

index = 1

# Plot the original image for reference
plt.subplot(rows_plot, cols_plot, index)
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.title("Original")
plt.axis("off")

# Use a fixed random generator so results are reproducible
rng = np.random.default_rng(42)

# Loop over different fractions
for frac in fractions:

    # Create a random mask:
    # Each frequency is kept with probability = frac
    # True (1) → keep, False (0) → remove
    mask = rng.random((rows, cols)) < frac

    # Apply the mask in frequency space
    # This keeps selected frequencies and zeroes out the rest
    F_filtered = F_shifted * mask

    # Transform back to pixel space
    # First undo fftshift, then apply inverse FFT
    img_rec_complex = np.fft.ifft2(np.fft.ifftshift(F_filtered))

    # Take real part (imaginary part is numerical noise)
    img_rec = np.real(img_rec_complex)

    # Move to next subplot
    index += 1

    # Display reconstructed image
    plt.subplot(rows_plot, cols_plot, index)
    plt.imshow(img_rec, cmap="gray", vmin=0, vmax=255)
    plt.title(f"{frac*100:.1f}% frequencies")
    plt.axis("off")

plt.tight_layout()
plt.show()

### Why does random sampling look noisy?

Because frequency importance is not uniform across the spectrum.

In natural images:
- low-frequency components carry most of the global structure
- high-frequency components refine details

Random sampling ignores this structure.

Even if we keep many frequencies, we may:
- miss important low-frequency components
- keep many irrelevant high-frequency components

As a result, the reconstruction lacks coherent structure and appears noisy.

---

### Key takeaway

It is not just the **number of frequencies** that matters, but **which frequencies** we keep.

A structured selection (like a low-pass filter) preserves the image much better than a random selection.

## Step 8 — Animate the reconstruction

A static grid is useful, but an animation makes the idea more intuitive.

Here we gradually increase `keep`. At each frame, the square mask in frequency space becomes larger, so more Fourier components are included in the reconstruction.

This shows the image being rebuilt from coarse, low-frequency structure to finer detail.

In [ ]:
# Values of keep used in the animation.
# Here we increase keep one step at a time from 1 to 100.
# Larger keep values include more frequencies and therefore more detail.
keep_values_anim = list(range(1, 101))

# Create a figure and axis for the animation.
fig, ax = plt.subplots(figsize=(5, 5))
ax.axis("off")

def reconstruct_lowpass(keep):
    """
    Reconstruct the image using a square low-pass Fourier mask.

    Parameters
    ----------
    keep : int
        Half-width of the square region kept around the center
        of the shifted Fourier spectrum.

    Returns
    -------
    img_rec : 2D NumPy array
        Reconstructed image using only the retained Fourier coefficients.
    """

    # Start with a mask of zeros: remove all frequencies by default.
    mask = np.zeros((rows, cols))

    # Keep a square block around the center of the spectrum.
    mask[crow-keep:crow+keep, ccol-keep:ccol+keep] = 1

    # Apply the mask in frequency space.
    F_filtered = F_shifted * mask

    # Reconstruct the image:
    # 1. Undo fftshift
    # 2. Apply inverse 2D FFT
    img_rec_complex = np.fft.ifft2(np.fft.ifftshift(F_filtered))

    # The image should be real-valued.
    # Small imaginary parts may appear due to numerical roundoff.
    img_rec = np.real(img_rec_complex)

    return img_rec

# Initialize the animation using the first reconstructed image.
first_img = reconstruct_lowpass(keep_values_anim[0])

im = ax.imshow(
    first_img,
    cmap="gray",
    vmin=0,
    vmax=255
)

ax.set_title(f"Fourier Reconstruction: keep = {keep_values_anim[0]}")

def update(frame):
    """
    Update function called by FuncAnimation for each frame.
    """

    # Convert frame index into a keep value.
    keep = keep_values_anim[frame]

    # Reconstruct image for this keep value.
    img_rec = reconstruct_lowpass(keep)

    # Update the displayed image and title.
    im.set_data(img_rec)
    ax.set_title(f"Fourier Reconstruction: keep = {keep}")

    return [im]

# Create the animation.
# interval is the delay between frames in milliseconds.
anim = FuncAnimation(
    fig,
    update,
    frames=len(keep_values_anim),
    interval=400,
    blit=False
)

# Save as GIF.
# This may take some time because each frame must be rendered.
anim.save(
    "low_pass_fourier_reconstruction.gif",
    writer=PillowWriter(fps=10)
)

# Optional: save as MP4.
# This requires ffmpeg to be installed.
# anim.save(
#     "low_pass_fourier_reconstruction.mp4",
#     writer=FFMpegWriter(fps=10)
# )

# Prevent matplotlib from showing a duplicate static figure.
plt.close(fig)

# Display the animation inside the notebook.
HTML(anim.to_jshtml())

### What the animation shows

At the beginning, only a tiny region near the center of the Fourier spectrum is used, so the image contains only broad, smooth structure.

As `keep` increases, the mask expands outward. More frequency components are added, and the reconstruction becomes sharper.

This makes the frequency-space idea visible: the image is not recovered all at once, but gradually, as more Fourier modes are included.

## Step 9 — Magnitude and phase

A Fourier coefficient is complex:

$$
F(u,v) = |F(u,v)| e^{i\phi(u,v)}
$$

This form separates each Fourier coefficient into two parts:

- magnitude $|F(u,v)|$: how strong that frequency component is  
- phase $\phi(u,v)$: how that component is aligned in space  

A key observation from image Fourier analysis is:

> The phase strongly influences the **spatial structure and alignment** of image features,  
> while the magnitude controls the **strength of each frequency component**.

If we reconstruct an image using only phase and discard the original magnitudes, we usually do not recover a photorealistic image. However, outlines, edges, and spatial structure can still remain visible.

On the other hand, using only magnitude and discarding phase often produces a smooth or unstructured result with little recognizable content.

This demonstrates that:

- magnitude tells us *how much* of each frequency is present  
- phase helps determine *where* those frequency patterns appear in the image  

In practice, the correct spatial arrangement — and therefore recognizability — depends strongly on the phase.

### 9.1 Basic phase-only and magnitude-only reconstruction

First, we reconstruct the image using only one part of the Fourier coefficients at a time.

- In the **phase-only case**, we keep the phase $\phi(u,v)$ and set all magnitudes to one.  
  This means all frequency components are given equal strength, but their spatial alignment is preserved.

- In the **magnitude-only case**, we keep the magnitude $|F(u,v)|$ and set all phases to zero.  
  This preserves how strong each frequency is, but removes the spatial alignment between them.

This version applies only a mild enhancement for visualization, so the result remains close to the raw reconstruction while making the structure easier to see.

In [ ]:
# --- Extract magnitude and phase from the Fourier spectrum ---
magnitude = np.abs(F_shifted)   # strength of each frequency
phase = np.angle(F_shifted)     # phase (spatial alignment)

# --- Phase-only reconstruction ---
# Set magnitude = 1 everywhere, keep only phase information
F_phase_only = np.exp(1j * phase)

# Reconstruct image
img_phase_only = np.fft.ifft2(np.fft.ifftshift(F_phase_only))

# Take real part (imaginary part is numerical noise)
img_phase_only = np.real(img_phase_only)

# --- Magnitude-only reconstruction ---
# Keep magnitude, but remove phase (set phase = 0)
F_mag_only = magnitude * np.exp(1j * 0)

# Reconstruct image
img_mag_only = np.fft.ifft2(np.fft.ifftshift(F_mag_only))
img_mag_only = np.real(img_mag_only)

# --- Helper: display signed values correctly ---
def show_signed(img, ax, title):
    # Use symmetric color limits so positive/negative variations are visible
    vmax = np.max(np.abs(img))
    ax.imshow(img, cmap="gray", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")

# --- Mild contrast enhancement ---
# Helps reveal structure without heavily distorting the image
def enhance(img):
    return np.sign(img) * np.sqrt(np.abs(img))

img_phase_vis = enhance(img_phase_only)
img_mag_vis = enhance(img_mag_only)

# --- Plot results ---
plt.figure(figsize=(12, 4))

# Original
ax = plt.subplot(1, 3, 1)
ax.imshow(img, cmap="gray")
ax.set_title("Original")
ax.axis("off")

# Phase-only
ax = plt.subplot(1, 3, 2)
show_signed(img_phase_vis, ax, "Phase Only (Enhanced)")

# Magnitude-only
ax = plt.subplot(1, 3, 3)
show_signed(img_mag_vis, ax, "Magnitude Only (Enhanced)")

plt.tight_layout()
plt.show()

### 9.2 Contrast-enhanced visualization

The phase-only reconstruction often has a very small dynamic range, so it can appear faint or nearly uniform when displayed directly.

This happens because, in the phase-only case, all frequency components are given equal magnitude. As a result, the overall intensity variations in the reconstructed image are much smaller than in the original.

To make the underlying structure easier to see, we apply a stronger contrast enhancement.

This step does **not** change the Fourier reconstruction itself. It only rescales the pixel values for visualization, making edges and spatial patterns more visible.

In [ ]:
# --- Extract magnitude and phase from the Fourier spectrum ---
# magnitude: strength of each frequency component
# phase: spatial alignment of each frequency component
magnitude = np.abs(F_shifted)
phase = np.angle(F_shifted)

# --- Phase-only reconstruction ---
# Here we discard the original magnitudes and keep only the phase.
# We set magnitude = 1 everywhere, so all frequencies contribute equally.
F_phase_only = np.exp(1j * phase)

# Reconstruct the image from phase-only spectrum
# 1. Undo fftshift
# 2. Apply inverse Fourier transform
img_phase_only_complex = np.fft.ifft2(np.fft.ifftshift(F_phase_only))

# Take real part (imaginary part is numerical noise)
img_phase_only = np.real(img_phase_only_complex)

# --- Magnitude-only reconstruction ---
# Here we keep the magnitude but remove phase information.
# Setting phase = 0 means all frequency components are aligned identically.
F_mag_only = magnitude * np.exp(1j * 0)

# Reconstruct image from magnitude-only spectrum
img_mag_only_complex = np.fft.ifft2(np.fft.ifftshift(F_mag_only))
img_mag_only = np.real(img_mag_only_complex)

# --- Strong contrast boost (KEY for visualization) ---
# Phase-only images often have very small dynamic range.
# This function rescales and enhances contrast so structure becomes visible.
def enhance(img):
    # Center around zero
    img = img - np.mean(img)

    # Normalize by standard deviation (avoid division by zero)
    img = img / (np.std(img) + 1e-8)

    # Apply nonlinear compression to enhance contrast
    # tanh squashes extreme values while keeping structure
    return np.tanh(img * 2)

# Apply enhancement for display
img_phase_vis = enhance(img_phase_only)
img_mag_vis = enhance(img_mag_only)

# --- Plot results ---
plt.figure(figsize=(12, 4))

# Original image
plt.subplot(1, 3, 1)
plt.imshow(img, cmap="gray")
plt.title("Original")
plt.axis("off")

# Phase-only (contrast enhanced)
plt.subplot(1, 3, 2)
plt.imshow(img_phase_vis, cmap="gray")
plt.title("Phase Only (Contrast Enhanced)")
plt.axis("off")

# Magnitude-only (contrast enhanced)
plt.subplot(1, 3, 3)
plt.imshow(img_mag_vis, cmap="gray")
plt.title("Magnitude Only (Contrast Enhanced)")
plt.axis("off")

plt.tight_layout()
plt.show()

### Interpretation

The phase-only image preserves the main geometry of the scene: edges, contours, and spatial layout remain visible. However, it does not reconstruct realistic shading, so the result looks more like an embossed or edge-enhanced version of the image rather than a natural photograph.

The magnitude-only image retains information about how strong each frequency component is, but without phase the components are not placed correctly in space. As a result, the reconstruction lacks coherent structure and becomes difficult to recognize.

This experiment highlights an important point:

- phase plays a dominant role in determining the **spatial arrangement** of features  
- magnitude determines the **strength** of each frequency component  

Both are necessary for a faithful reconstruction, but the recognizability of the image depends strongly on the phase.

In other words, magnitude tells us how much of each pattern exists, but phase tells us where those patterns belong.

## Step 10 — What we learned

This tutorial demonstrated the following ideas:

1. A grayscale image can be treated as a 2D function $f(x,y)$.
2. The 2D DFT represents the image as a collection of frequency components.
3. The inverse DFT reconstructs the image from those components.
4. Keeping only low frequencies gives a smooth approximation of the image.
5. Adding more frequencies restores sharper details and fine structure.
6. Random frequency sampling is inefficient because important information is not uniformly distributed in frequency space.
7. Phase plays a crucial role in preserving the spatial structure of the image.

So yes, a face can be rebuilt from Fourier components.

More precisely, a digital image can be reconstructed as a weighted sum of discrete 2D sinusoidal basis functions, where each component contributes a specific frequency, strength, and spatial alignment.

## Optional exercises

Try these variations to deepen your understanding:

1. Use your own image and repeat the experiment.
2. Try different types of images, such as text, buildings, or landscapes.
3. Change `keep_values` and observe how quickly details return.
4. Replace the square mask with a circular mask.
5. Extend the experiment to color images by applying the transform separately to each RGB channel.

The best way to understand Fourier reconstruction is to change one thing at a time and observe what happens.

## Notes on images, usage, and ethics

This tutorial uses an image for demonstration purposes. If you are using your own images, please keep the following in mind:

- Make sure you have the right to use and share the image.
- Avoid using copyrighted images without permission.
- If using publicly available images, prefer those with open licenses (e.g., Creative Commons).

When working with images of people:

- Be mindful of privacy and consent.
- Avoid using identifiable images without appropriate permission, especially in public repositories.

This tutorial is intended for educational purposes. The techniques shown here (Fourier analysis and reconstruction) are widely used in image processing, compression, and scientific computing.